# Structured RAG over a mixed PDF with Azure Content Understanding + Azure AI Search

**Goal:** turn `mixed_content.pdf` (text + tables, one table split across a page break) into search-ready, structured knowledge, then let a *small* model answer questions with references (section + page) — without ever reading the whole PDF.

**What this notebook shows, step by step**

| Part | What happens | Content Understanding (CU) value shown |
| --- | --- | --- |
| 2 | `prebuilt-layout`: OCR + layout → Markdown | No LLM tokens, billed per page |
| 3 | Custom analyzer with a field schema | Typed fields, classification, confidence, grounding, cross-page tables as one list |
| 4 | Post-processing into search records | Heading paths, page numbers, atomic tables, row-level facts |
| 5–6 | Azure AI Search index + retrieval | Hybrid / semantic search with metadata filters |
| 7–8 | Grounded answers and a tool-calling agent | Small model + small context + citations |
| 9 | Measured comparison | Tokens, latency and correctness vs. sending the whole PDF to a premium model |

## 0.1 Why Content Understanding? (concept check)

| Claim | Reality |
| --- | --- |
| "CU saves LLM tokens" | **Content extraction** (`prebuilt-read`, `prebuilt-layout`) uses **no LLM** – billed per page. **Generative features** (field extract / generate / classify, figure description, segmentation) **do** use tokens on *your* Foundry deployment, plus a CU *contextualization* charge. |
| "CU is faster" | Analysis is an async long-running operation (seconds per document). The speed win is **at query time**: retrieval (~100 ms) + a small model reading a small context. |
| "Premium models burn tokens on the same task" | Sending a PDF to a vision model costs text **and page-image** tokens **on every question**. With CU + Search you pay ingestion **once**, then each question sends only the top-k relevant chunks to a mini model. |
| Extra value | Schema-consistent output, confidence scores, source grounding (page + region), normalized numbers, OCR for scans, scales beyond any context window. |

**Where the money goes**

| Stage | Frequency | Meters |
| --- | --- | --- |
| Content extraction | once per page | CU Document Minimal / Basic / Standard |
| Field extraction | once per document | CU contextualization tokens + LLM tokens on your deployment |
| Indexing | once per chunk | Embedding tokens |
| Question answering | **every query** | 1 query embedding + small-model tokens on top-k chunks |

## 0.2 Architecture

> Mermaid diagrams render natively in JupyterLab 4.1+; in VS Code install the *Markdown Preview Mermaid Support* extension.

```mermaid
flowchart LR
  subgraph LOCAL["Local - VS Code + Jupyter (adi kernel)"]
    NB["02_structured_rag_mixed_content.ipynb"]
    PDF[("data/mixed_content.pdf")]
    CACHE[("data/output/*.json<br/>cached CU results")]
    ENV[".env + az login"]
  end
  ENTRA["Microsoft Entra ID<br/>DefaultAzureCredential"]
  subgraph FOUNDRY["Microsoft Foundry resource"]
    subgraph CU["Content Understanding (GA API 2025-11-01)"]
      LAYOUT["prebuilt-layout<br/>OCR + layout, no LLM"]
      CUSTOM["mixedContentReport<br/>custom analyzer (base: prebuilt-document)<br/>markdown + fields + confidence"]
    end
    subgraph MODELS["Model deployments"]
      MINI["gpt-5.4-mini<br/>CU fields / RAG answers / agent"]
      PREM["gpt-5.4<br/>baseline only"]
      EMB["text-embedding-3-large<br/>3072-d vectors"]
    end
  end
  subgraph SEARCH["Azure AI Search (Free tier)"]
    IDX[("mixed-content-rag index<br/>text / table / table_row / summary")]
    SEM["Semantic ranker<br/>free monthly allowance"]
  end
  ENV --> ENTRA
  NB -- "token" --> ENTRA
  PDF --> NB
  NB -- "analyze (LRO)" --> LAYOUT
  NB -- "create + analyze" --> CUSTOM
  CUSTOM -- "field extraction tokens" --> MINI
  NB <--> CACHE
  NB -- "embed chunks / queries" --> EMB
  NB -- "upload / hybrid query" --> IDX
  IDX --- SEM
  NB -- "answer with top-k chunks" --> MINI
  NB -- "baseline: whole PDF" --> PREM
```

## 0.3 Ingestion pipeline (one-time per document)

```mermaid
flowchart TD
  A["mixed_content.pdf<br/>text + 2 tables, one split across pages"] --> B{"USE_CACHE and<br/>cached JSON exists?"}
  B -- yes --> C["Load AnalysisResult from JSON"]
  B -- no --> D["CU analyze with mixedContentReport"]
  D --> E["Save result + poller.usage to data/output"]
  C --> F["DocumentContent"]
  E --> F
  F --> G["markdown<br/>headings, markdown tables,<br/>PageBreak / PageHeader comments"]
  F --> H["structure<br/>pages.spans / paragraphs.role / tables.span"]
  F --> I["fields<br/>Title / DocumentType / Summary / Keywords<br/>Sections[] / IndustryAIAdoption[]<br/>CloudProviderMarketShare[] / Recommendations[]"]
  G & H --> J["Merge cross-page tables<br/>same columns + repeated header + only page markers between"]
  J --> K["Split by heading path H1 > H2 > H3<br/>keeping character offsets"]
  K --> L["Tables -> atomic table chunks"]
  K --> M["Text -> split by size,<br/>strip page markers, rejoin split paragraphs"]
  L & M --> N["Offsets -> page_start / page_end"]
  I --> O["Heading -> topic lookup"]
  I --> P["Extra records<br/>1 summary record<br/>1 table_row record per table row"]
  N & O --> Q["Records + contextual header<br/>Doc | Section path | pages"]
  Q & P --> R["Embed with text-embedding-3-large"]
  R --> S[("Upload to mixed-content-rag")]
```

## 0.4 Query time (agent with a search tool)

```mermaid
sequenceDiagram
  autonumber
  actor U as User
  participant NB as Notebook agent loop
  participant LLM as gpt-5.4-mini (Responses API)
  participant T as search_document tool
  participant EMB as text-embedding-3-large
  participant S as AI Search (hybrid + semantic)
  U->>NB: Which cloud provider grew fastest YoY?
  NB->>LLM: question + tool schema (query, chunk_type, topic)
  LLM-->>NB: function_call search_document
  NB->>T: execute
  T->>EMB: embed query
  EMB-->>T: vector
  T->>S: BM25 + vector (RRF) + semantic rerank + filter
  S-->>T: top-k records with section_path and pages
  T-->>NB: numbered sources [1]..[k]
  NB->>LLM: function_call_output (small context only)
  LLM-->>NB: answer with [n] citations
  NB-->>U: Answer + References (section, pages)
```

## 0.5 Measured comparison (Part 9)

```mermaid
flowchart LR
  Q["Eval set<br/>questions + expected keywords"] --> A & B & C
  A["A: Full PDF (text + page images)<br/>to gpt-5.4 every question"]
  B["B: Full CU markdown, no retrieval<br/>to gpt-5.4-mini"]
  C["C: CU + AI Search top-k<br/>to gpt-5.4-mini"]
  A & B & C --> M["Per question: input / output / reasoning tokens,<br/>latency, keyword correctness"]
  ING["One-time ingestion<br/>CU pages + contextualization + LLM tokens<br/>+ embedding tokens"] --> P
  M --> P["Cost projection<br/>PRICES you fill in x queries x pages"]
```

## 0.6 Models used and why

| Deployment | Used for | Why |
| --- | --- | --- |
| `gpt-5.4-mini` | CU field extraction, RAG answers, agent | Supported by CU, cheap and fast — the "simple LLM" |
| `gpt-5.4` | Baseline A only | Premium comparison point |
| `text-embedding-3-large` | Chunk + query vectors; CU analyzer embedding mapping | CU only spends embedding tokens when you add labeled samples (not here) |

## 0.7 Prerequisites

1. **Foundry resource** with Content Understanding and the three deployments above. Your user needs `Cognitive Services User`.
2. **Azure AI Search** (Free tier is fine: 50 MB, max 3 indexes). *Settings → Keys → API access control* = *Both* or *RBAC*. Roles: `Search Service Contributor` + `Search Index Data Contributor`. Semantic ranker: free plan (monthly allowance, region-dependent).
3. `.env` (copy from `.env.sample`): `AZURE_OPENAI_CHAT_DEPLOYMENT`, `AZURE_OPENAI_PREMIUM_DEPLOYMENT`, `CU_COMPLETION_MODEL` are new for this notebook.
4. `az login`, kernel `adi`.

## 1. Setup

### 1.1 Configuration and flags
`USE_CACHE=True` reuses saved CU results from `data/output/` so you can re-run or present without new CU calls. Delete the JSON files (or set it to `False`) to analyze again.

In [1]:
import base64
import json
import os
import re
import time
from pathlib import Path
from statistics import mean

from dotenv import find_dotenv, load_dotenv
from IPython.display import Markdown, display

load_dotenv(find_dotenv(usecwd=True))


def require_env(name: str) -> str:
    value = os.getenv(name)
    if not value:
        raise RuntimeError(f"Missing environment variable '{name}'. See .env.sample.")
    return value


CU_ENDPOINT = require_env("CONTENTUNDERSTANDING_ENDPOINT")
AOAI_ENDPOINT = require_env("AZURE_OPENAI_ENDPOINT")
EMBEDDING_DEPLOYMENT = require_env("AZURE_OPENAI_EMBEDDING_DEPLOYMENT")
CHAT_DEPLOYMENT = require_env("AZURE_OPENAI_CHAT_DEPLOYMENT")  # small model, e.g. gpt-5.4-mini
PREMIUM_DEPLOYMENT = require_env("AZURE_OPENAI_PREMIUM_DEPLOYMENT")  # baseline only, e.g. gpt-5.4
SEARCH_ENDPOINT = require_env("AZURE_SEARCH_ENDPOINT")
SEARCH_API_KEY = os.getenv("AZURE_SEARCH_API_KEY")  # optional last-resort fallback

# CU analyzers reference model NAMES; update_defaults (1.3) maps them to deployment names.
CU_COMPLETION_MODEL = os.getenv("CU_COMPLETION_MODEL", "gpt-5.4-mini")
CU_EMBEDDING_MODEL = "text-embedding-3-large"

DOC_PATH = Path("../data/mixed_content.pdf")
OUTPUT_DIR = Path("../data/output")
assert DOC_PATH.exists(), f"File not found: {DOC_PATH.resolve()}"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

ANALYZER_ID = "mixedContentReport"
INDEX_NAME = "mixed-content-rag"
EMBEDDING_DIMENSIONS = 3072
REASONING_EFFORT = "low"  # None for non-reasoning models such as gpt-4o

USE_CACHE = True
USE_SEMANTIC_RANKER = True  # False -> hybrid only (saves the free monthly semantic allowance)

### 1.2 Clients
One `DefaultAzureCredential` (your `az login`) backs CU, the OpenAI `/openai/v1/` endpoint and Azure AI Search. The search key is used only if you set `AZURE_SEARCH_API_KEY`.

**Corporate proxy note:** the search clients use a custom TLS transport. A TLS-inspecting proxy (Netskope) re-signs traffic with a root CA that has no `keyUsage` extension, and Python 3.13 / urllib3 2.x reject it under strict X.509 checks (`CA cert does not include key usage extension`). The transport trusts the CA bundle from `REQUESTS_CA_BUNDLE` and clears only `VERIFY_X509_STRICT`; certificate chain and hostname verification stay on. Pass `transport=search_transport` to any other Azure SDK client that hits the same error.

In [24]:
import ssl

import requests
from azure.ai.contentunderstanding import ContentUnderstandingClient
from azure.core.credentials import AzureKeyCredential
from azure.core.pipeline.transport import RequestsTransport
from azure.identity import DefaultAzureCredential, get_bearer_token_provider
from azure.search.documents import SearchClient
from azure.search.documents.indexes import SearchIndexClient
from openai import OpenAI
from requests.adapters import HTTPAdapter

credential = DefaultAzureCredential()
token_provider = get_bearer_token_provider(credential, "https://cognitiveservices.azure.com/.default")

cu_client = ContentUnderstandingClient(endpoint=CU_ENDPOINT, credential=credential)
openai_client = OpenAI(base_url=f"{AOAI_ENDPOINT.rstrip('/')}/openai/v1/", api_key=token_provider)

# Corporate TLS proxy: its root CA lacks keyUsage, which strict X.509 checks reject.
# Certificates and hostnames are still verified; only the strict RFC 5280 flag is cleared.
tls_context = ssl.create_default_context(cafile=os.getenv("REQUESTS_CA_BUNDLE"))
tls_context.verify_flags &= ~ssl.VERIFY_X509_STRICT


class ProxyCATLSAdapter(HTTPAdapter):
    def init_poolmanager(self, *args, **kwargs):
        kwargs["ssl_context"] = tls_context
        return super().init_poolmanager(*args, **kwargs)


search_session = requests.Session()
search_session.mount("https://", ProxyCATLSAdapter())
search_transport = RequestsTransport(session=search_session, session_owner=False)

search_credential = AzureKeyCredential(SEARCH_API_KEY) if SEARCH_API_KEY else credential
index_client = SearchIndexClient(endpoint=SEARCH_ENDPOINT, credential=search_credential, transport=search_transport)
search_client = SearchClient(endpoint=SEARCH_ENDPOINT, index_name=INDEX_NAME, credential=search_credential, transport=search_transport)

Two small display helpers used throughout: `show_table()` renders a list of dicts as a Markdown table, and `pages_of()` reads page numbers from CU `source` strings such as `D(3,0.9,1.2,...)`.

In [3]:
def show_table(rows: list[dict], columns: list[str] | None = None) -> None:
    if not rows:
        print("(no rows)")
        return
    columns = columns or list(rows[0].keys())

    def cell(value) -> str:
        if isinstance(value, (list, tuple)):
            value = ", ".join(map(str, value))
        return str(value).replace("|", "\\|").replace("\n", " ")

    lines = ["| " + " | ".join(columns) + " |", "| " + " | ".join("---" for _ in columns) + " |"]
    lines += ["| " + " | ".join(cell(row.get(c, "")) for c in columns) + " |" for row in rows]
    display(Markdown("\n".join(lines)))


def pages_of(source: str | None) -> list[int]:
    return sorted({int(p) for p in re.findall(r"D\((\d+),", source or "")})

### 1.3 Map CU models to your deployments (one-time per Foundry resource)
CU calls *your* deployments for generative work. Keys are **model names** (or `prebuilt-analyzer-*` aliases), values are **deployment names**. This setting is resource-wide, so it also affects `prebuilt-documentSearch` in notebook 01.

In [4]:
defaults = cu_client.update_defaults(
    model_deployments={
        CU_COMPLETION_MODEL: CHAT_DEPLOYMENT,
        CU_EMBEDDING_MODEL: EMBEDDING_DEPLOYMENT,
        "prebuilt-analyzer-completion": CHAT_DEPLOYMENT,
        "prebuilt-analyzer-completion-mini": CHAT_DEPLOYMENT,
        "prebuilt-analyzer-embedding": EMBEDDING_DEPLOYMENT,
    }
)
print(json.dumps(defaults.model_deployments, indent=2))

{
  "gpt-5.4": "gpt-5.4",
  "gpt-5-mini": "gpt-5-mini",
  "text-embedding-3-large": "text-embedding-3-large",
  "prebuilt-analyzer-completion": "gpt-5.4-mini",
  "prebuilt-analyzer-completion-mini": "gpt-5.4-mini",
  "prebuilt-analyzer-embedding": "text-embedding-3-large",
  "gpt-4.1-mini": "gpt-4.1-mini",
  "gpt-5.4-mini": "gpt-5.4-mini"
}


## 2. Content extraction only: `prebuilt-layout` (no LLM)

### 2.1 Analyze with caching
`analyze_with_cache()` runs an analyzer once, then saves the result, the **usage** reported by the service (`poller.usage`) and the elapsed time to `data/output/`.

In [5]:
from azure.ai.contentunderstanding.models import AnalysisResult


def analyze_with_cache(analyzer_id: str) -> tuple[AnalysisResult, dict]:
    cache_file = OUTPUT_DIR / f"{DOC_PATH.stem}.{analyzer_id}.json"
    if USE_CACHE and cache_file.exists():
        cached = json.loads(cache_file.read_text(encoding="utf-8"))
        print(f"Loaded cached result: {cache_file.name}")
        return AnalysisResult(cached["result"]), {"usage": cached["usage"], "seconds": cached["seconds"]}

    start = time.perf_counter()
    poller = cu_client.begin_analyze_binary(analyzer_id=analyzer_id, binary_input=DOC_PATH.read_bytes())
    result = poller.result()
    meta = {"usage": poller.usage.as_dict() if poller.usage else {}, "seconds": round(time.perf_counter() - start, 2)}
    cache_file.write_text(json.dumps({"result": result.as_dict(), **meta}, indent=2, default=str), encoding="utf-8")
    print(f"Analyzed with '{analyzer_id}' in {meta['seconds']}s (operation {poller.operation_id})")
    return result, meta

In [6]:
layout_result, layout_meta = analyze_with_cache("prebuilt-layout")
layout_doc = layout_result.contents[0]

print(f"Pages {layout_doc.start_page_number}-{layout_doc.end_page_number} | "
      f"paragraphs: {len(layout_doc.paragraphs or [])} | tables: {len(layout_doc.tables or [])} | "
      f"markdown chars: {len(layout_doc.markdown)}")
print("Usage:", json.dumps(layout_meta["usage"], indent=2))

Loaded cached result: mixed_content.prebuilt-layout.json
Pages 1-3 | paragraphs: 87 | tables: 2 | markdown chars: 5262
Usage: {
  "documentPagesStandard": 3
}


**Look at the usage:** only pages are counted — no contextualization and no LLM tokens. This is the cheapest, deterministic part of CU.

### 2.2 The raw Markdown
Note the HTML comments CU inserts: `<!-- PageBreak -->`, `<!-- PageHeader=... -->`, `<!-- PageNumber=... -->`. They are useful for page mapping but noise for an LLM.

In [7]:
print(layout_doc.markdown[:4000])

# 2024 Technology Trends Report


## Executive Summary

This report provides an analysis of the most significant technology trends shaping the industry
in 2024. From the rapid adoption of generative AI to advances in quantum computing, these
trends are reshaping how businesses operate and compete. Understanding these trends is
essential for strategic planning and technology investment decisions.


## Key Findings

. Generative Al adoption has increased by 340% year-over-year across enterprise
organizations.

. Cloud-native architectures now power 78% of new application deployments.

. Cybersecurity spending has reached $188 billion globally, driven by increasing threat
sophistication.

. Edge computing deployments grew 45% as organizations seek lower latency and
data sovereignty.

. Sustainability-focused technology initiatives now represent 23% of IT budgets on
average.

. Low-code and no-code platforms are used by 65% of organizations for at least some
development.

. API-first strat

### 2.3 Document structure
Every paragraph carries a semantic `role` (`title`, `sectionHeading`, `pageHeader`, ...) and a `source` with its page and bounding polygon. Body paragraphs have no role.

In [8]:
show_table(
    [
        {"#": i, "role": p.role or "body", "pages": pages_of(p.source), "text": p.content[:90]}
        for i, p in enumerate(layout_doc.paragraphs or [])
    ]
)

| # | role | pages | text |
| --- | --- | --- | --- |
| 0 | SemanticRole.SECTION_HEADING | 1 | 2024 Technology Trends Report |
| 1 | SemanticRole.SECTION_HEADING | 1 | Executive Summary |
| 2 | body | 1 | This report provides an analysis of the most significant technology trends shaping the ind |
| 3 | SemanticRole.SECTION_HEADING | 1 | Key Findings |
| 4 | body | 1 | . Generative Al adoption has increased by 340% year-over-year across enterprise organizati |
| 5 | body | 1 | . Cloud-native architectures now power 78% of new application deployments. |
| 6 | body | 1 | . Cybersecurity spending has reached $188 billion globally, driven by increasing threat so |
| 7 | body | 1 | . Edge computing deployments grew 45% as organizations seek lower latency and data soverei |
| 8 | body | 1 | . Sustainability-focused technology initiatives now represent 23% of IT budgets on average |
| 9 | body | 1 | . Low-code and no-code platforms are used by 65% of organizations for at least some develo |
| 10 | body | 1 | . API-first strategies have been adopted by 82% of enterprises surveyed. |
| 11 | SemanticRole.SECTION_HEADING | 1 | Generative AI in the Enterprise |
| 12 | body | 1 | Generative AI has emerged as the defining technology of 2024. Large language models (LLMs) |
| 13 | body | 1 | Organizations are investing heavily in data infrastructure to support their AI initiatives |
| 14 | body | 2 | modern RAG deployments. The ability to accurately extract and structure information from u |
| 15 | SemanticRole.SECTION_HEADING | 2 | AI Adoption by Industry |
| 16 | body | 2 | Industry |
| 17 | body | 2 | Adoption Rate |
| 18 | body | 2 | Primary Use Case |
| 19 | body | 2 | Avg. ROI |
| 20 | body | 2 | Financial Services |
| 21 | body | 2 | 89% |
| 22 | body | 2 | Fraud Detection & Risk Analysis |
| 23 | body | 2 | 340% |
| 24 | body | 2 | Healthcare |
| 25 | body | 2 | 76% |
| 26 | body | 2 | Diagnostic Assistance |
| 27 | body | 2 | 280% |
| 28 | body | 2 | Manufacturing |
| 29 | body | 2 | 72% |
| 30 | body | 2 | Predictive Maintenance |
| 31 | body | 2 | 310% |
| 32 | body | 2 | Retail |
| 33 | body | 2 | 81% |
| 34 | body | 2 | Personalization & Recommendations |
| 35 | body | 2 | 250% |
| 36 | body | 2 | Technology |
| 37 | body | 2 | 94% |
| 38 | body | 2 | Code Generation & Testing |
| 39 | body | 2 | 420% |
| 40 | body | 2 | Education |
| 41 | body | 2 | 58% |
| 42 | body | 2 | Personalized Learning |
| 43 | body | 2 | 180% |
| 44 | SemanticRole.SECTION_HEADING | 2 | Cloud and Infrastructure Trends |
| 45 | body | 2 | The cloud computing landscape continues to evolve rapidly. Multi-cloud and hybrid-cloud st |
| 46 | SemanticRole.SECTION_HEADING | 2 | Top Cloud Priorities for 2024 |
| 47 | body | 2 | . Cost optimization and FinOps practices to manage growing cloud spend. |
| 48 | body | 2 | . Multi-cloud governance and consistent security policies across providers. |
| 49 | body | 2 | . Serverless and event-driven architectures for new microservices. |
| 50 | body | 2 | . Infrastructure as Code (laC) maturity using Terraform, Pulumi, and CDK. |
| 51 | body | 2 | . Green computing initiatives to reduce carbon footprint of cloud workloads. |
| 52 | SemanticRole.SECTION_HEADING | 2 | Cloud Provider Market Share |
| 53 | body | 2 | Provider |
| 54 | body | 2 | Market Share |
| 55 | body | 2 | YoY Growth |
| 56 | body | 2 | Key Strength |
| 57 | body | 2 | AWS |
| 58 | body | 2 | 31% |
| 59 | body | 2 | +12% |
| 60 | body | 2 | Breadth of services |
| 61 | body | 2 | Microsoft Azure |
| 62 | body | 2 | 25% |
| 63 | body | 2 | +18% |
| 64 | body | 2 | Enterprise integration |
| 65 | body | 3 | Provider |
| 66 | body | 3 | Market Share |
| 67 | body | 3 | YoY Growth |
| 68 | body | 3 | Key Strength |
| 69 | body | 3 | Google Cloud |
| 70 | body | 3 | 11% |
| 71 | body | 3 | +22% |
| 72 | body | 3 | AI/ML capabilities |
| 73 | body | 3 | Others |
| 74 | body | 3 | 33% |
| 75 | body | 3 | +8% |
| 76 | body | 3 | Specialized offerings |
| 77 | SemanticRole.SECTION_HEADING | 3 | Cybersecurity Landscape |
| 78 | body | 3 | The cybersecurity threat landscape has become increasingly complex. Ransomware attacks con |
| 79 | body | 3 | Supply chain security has emerged as a critical concern following high-profile incidents.  |
| 80 | SemanticRole.SECTION_HEADING | 3 | Recommendations |
| 81 | body | 3 | . Invest in RAG-based Al architectures for knowledge management applications. |
| 82 | body | 3 | . Establish robust document parsing pipelines to feed Al systems with high-quality data. |
| 83 | body | 3 | . Adopt a multi-cloud strategy with consistent governance and security policies. |
| 84 | body | 3 | . Prioritize cybersecurity investments, especially in zero-trust and supply chain security |
| 85 | body | 3 | . Build internal Al literacy through training programs and centers of excellence. |
| 86 | body | 3 | . Evaluate sustainability metrics for all technology decisions and vendor selections. |

In [9]:
show_table(
    [
        {
            "table": i,
            "rows x cols": f"{t.row_count} x {t.column_count}",
            "pages": pages_of(t.source),
            "header": [c.content for c in t.cells if c.row_index == 0],
        }
        for i, t in enumerate(layout_doc.tables or [])
    ]
)

| table | rows x cols | pages | header |
| --- | --- | --- | --- |
| 0 | 7 x 4 | 2 | Industry, Adoption Rate, Primary Use Case, Avg. ROI |
| 1 | 6 x 4 | 2, 3 | Provider, Market Share, YoY Growth, Key Strength |

### 2.4 The two cross-page problems in this PDF
1. A paragraph that starts on page 1 (*"...form the backbone of"*) and continues on page 2 (*"modern RAG deployments..."*).
2. The *Cloud Provider Market Share* table: two rows on page 3, a repeated header and two more rows on page 4.

The context around each page break shows how layout represents them.

In [10]:
for match in re.finditer(r"<!-- PageBreak -->", layout_doc.markdown):
    start, end = max(0, match.start() - 250), match.end() + 350
    print("=" * 100)
    print(layout_doc.markdown[start:end])

f
generative models with the accuracy of information retrieval.

Organizations are investing heavily in data infrastructure to support their AI initiatives. Vector
databases, embedding pipelines, and document processing systems form the backbone of

<!-- PageBreak -->

modern RAG deployments. The ability to accurately extract and structure information from
unstructured documents is a critical capability that directly impacts AI application quality.


### AI Adoption by Industry


<table>
<tr>
<th>Industry</th>
<th>Adoption Rate</th>
<th>Primary Use Case</th>
<th>Avg. ROI</th>
</tr>
<tr>
<td>Financial Services</


## 3. Custom analyzer: structured output with a field schema

A custom analyzer extends the base `prebuilt-document` analyzer. One analyze call returns **both** the layout Markdown and the fields you define.

| Method | Behaviour | Used here for |
| --- | --- | --- |
| `extract` | Value as it appears in the document (documents only; always grounded) | Title, table rows, recommendations |
| `generate` | Value written by the model from the content | Summary, keywords, per-section info |
| `classify` | One value from a fixed `enum` | Document type, section topic |

`estimate_field_source_and_confidence=True` returns a **confidence** (0–1) and a **source** (page + region) per value — the basis for citations and human-review routing. It also adds tokens.

### 3.1 Field schema
Descriptions act as mini-prompts: be specific, mention column names and formats.

In [11]:
from azure.ai.contentunderstanding.models import (
    ContentAnalyzer,
    ContentAnalyzerConfig,
    ContentFieldDefinition as FieldDef,
    ContentFieldSchema,
)

TOPICS = {
    "GenerativeAI": "Generative AI, LLMs, RAG and AI adoption",
    "CloudInfrastructure": "Cloud computing, multi-cloud, Kubernetes and cloud providers",
    "Cybersecurity": "Security threats, zero trust and supply chain security",
    "Recommendations": "Recommendations, next steps or calls to action",
    "General": "Introductions, executive summaries and anything else",
}

field_schema = ContentFieldSchema(
    name="IndustryReport",
    description="Structured view of a technology industry report for search and analytics.",
    fields={
        "Title": FieldDef(type="string", method="extract", description="Main title of the report as printed at the top of the first page."),
        "DocumentType": FieldDef(
            type="string",
            method="classify",
            description="Kind of document.",
            enum=["IndustryReport", "ResearchPaper", "Contract", "Invoice", "Policy", "Other"],
            enum_descriptions={
                "IndustryReport": "Market or technology trends report with statistics",
                "ResearchPaper": "Academic or scientific paper",
                "Contract": "Legal agreement between parties",
                "Invoice": "Bill for goods or services",
                "Policy": "Internal policy or guideline",
                "Other": "Anything else",
            },
        ),
        "Summary": FieldDef(type="string", method="generate", description="Three-sentence summary of the whole report covering every major section."),
        "Keywords": FieldDef(
            type="array", method="generate",
            description="5 to 10 short keywords or key phrases that describe the report.",
            item_definition=FieldDef(type="string"),
        ),
        "Sections": FieldDef(
            type="array", method="generate",
            description="One entry per heading in the report (including table titles), in reading order.",
            item_definition=FieldDef(
                type="object",
                properties={
                    "Heading": FieldDef(type="string", description="Heading text exactly as printed."),
                    "Topic": FieldDef(type="string", description="Main topic of the section.", enum=list(TOPICS), enum_descriptions=TOPICS),
                    "StartPage": FieldDef(type="integer", description="Page number where the heading appears."),
                    "Summary": FieldDef(type="string", description="One-sentence summary of the section."),
                },
            ),
        ),
        "IndustryAIAdoption": FieldDef(
            type="array", method="extract",
            description="Every row of the 'AI Adoption by Industry' table.",
            item_definition=FieldDef(
                type="object",
                properties={
                    "Industry": FieldDef(type="string", description="'Industry' column."),
                    "AdoptionRate": FieldDef(type="number", description="'Adoption Rate' column as a percent number, e.g. 89% -> 89."),
                    "PrimaryUseCase": FieldDef(type="string", description="'Primary Use Case' column."),
                    "AvgROI": FieldDef(type="number", description="'Avg. ROI' column as a percent number, e.g. 340% -> 340."),
                },
            ),
        ),
        "CloudProviderMarketShare": FieldDef(
            type="array", method="extract",
            description="Every row of the 'Cloud Provider Market Share' table. The table continues on the next page with a repeated header; include each provider once.",
            item_definition=FieldDef(
                type="object",
                properties={
                    "Provider": FieldDef(type="string", description="'Provider' column."),
                    "MarketShare": FieldDef(type="number", description="'Market Share' column as a percent number, e.g. 31% -> 31."),
                    "YoYGrowth": FieldDef(type="number", description="'YoY Growth' column as a percent number, e.g. +12% -> 12."),
                    "KeyStrength": FieldDef(type="string", description="'Key Strength' column."),
                },
            ),
        ),
        "Recommendations": FieldDef(
            type="array", method="extract",
            description="Each bullet point under the 'Recommendations' heading.",
            item_definition=FieldDef(type="string"),
        ),
    },
)

### 3.2 Create the analyzer
- `table_format="markdown"`: compact tables for chunks and prompts (`"html"` keeps merged cells).
- Figure description/analysis are off: this PDF has no figures, so no tokens are spent there.
- `models` uses model **names**; 1.3 mapped them to your deployments.

`allow_replace=True` makes the cell safe to re-run. If the service rejects part of the schema, the error message names the field (fallback: set that field's `method` to `"generate"`).

In [12]:
analyzer = ContentAnalyzer(
    base_analyzer_id="prebuilt-document",
    description="Industry report: layout markdown + structured fields for RAG",
    config=ContentAnalyzerConfig(
        enable_ocr=True,
        enable_layout=True,
        return_details=True,
        table_format="markdown",
        enable_figure_description=False,
        enable_figure_analysis=False,
        estimate_field_source_and_confidence=True,
    ),
    field_schema=field_schema,
    models={"completion": CU_COMPLETION_MODEL, "embedding": CU_EMBEDDING_MODEL},
)

created = cu_client.begin_create_analyzer(analyzer_id=ANALYZER_ID, resource=analyzer, allow_replace=True).result()
print(f"Analyzer '{created.analyzer_id}' ready with fields: {list(created.field_schema.fields)}")

Analyzer 'mixedContentReport' ready with fields: ['Title', 'DocumentType', 'Summary', 'Keywords', 'Sections', 'IndustryAIAdoption', 'CloudProviderMarketShare', 'Recommendations']


### 3.3 Analyze the PDF with the custom analyzer
Compare this usage with 2.1: now there are **contextualization tokens** and **LLM tokens** (on your `gpt-5.4-mini` deployment) on top of the pages.

In [13]:
cu_result, cu_meta = analyze_with_cache(ANALYZER_ID)
doc = cu_result.contents[0]
fields = doc.fields or {}

show_table(
    [
        {"analyzer": "prebuilt-layout", "seconds": layout_meta["seconds"], **layout_meta["usage"]},
        {"analyzer": ANALYZER_ID, "seconds": cu_meta["seconds"], **cu_meta["usage"]},
    ]
)

Loaded cached result: mixed_content.mixedContentReport.json


| analyzer | seconds | documentPagesStandard |
| --- | --- | --- |
| prebuilt-layout | 3.78 | 3 |
| mixedContentReport | 27.06 | 3 |

### 3.4 Read the fields
`to_plain()` turns CU field objects into plain Python values; `array_rows()` flattens a table field into rows with the lowest sub-field confidence and the source pages.

In [14]:
def to_plain(field):
    if field is None:
        return None
    if field.type == "array":
        return [to_plain(item) for item in field.value_array or []]
    if field.type == "object":
        return {name: to_plain(sub) for name, sub in (field.value_object or {}).items()}
    return field.value


def array_rows(name: str) -> list[dict]:
    rows = []
    for item in (fields[name].value_array or []) if name in fields else []:
        props = item.value_object or {}
        confidences = [sub.confidence for sub in props.values() if sub.confidence is not None]
        rows.append(
            {
                **{key: to_plain(sub) for key, sub in props.items()},
                "min_confidence": round(min(confidences), 3) if confidences else None,
                "pages": sorted({p for sub in props.values() for p in pages_of(sub.source)}),
            }
        )
    return rows


show_table(
    [
        {"field": name, "value": to_plain(fields.get(name)), "confidence": getattr(fields.get(name), "confidence", None), "pages": pages_of(getattr(fields.get(name), "source", None))}
        for name in ["Title", "DocumentType", "Summary", "Keywords", "Recommendations"]
    ]
)

| field | value | confidence | pages |
| --- | --- | --- | --- |
| Title | 2024 Technology Trends Report | 0.835 | 1 |
| DocumentType | IndustryReport | 0.835 | 1 |
| Summary | This report surveys major technology trends in 2024, highlighting generative AI adoption, cloud and infrastructure shifts, and an increasingly complex cybersecurity landscape. It presents industry AI adoption rates, cloud provider market share, and key priorities such as cost optimization, governance, and green computing. The report concludes with recommendations to invest in RAG-based AI, strengthen document pipelines, adopt multi-cloud governance, and prioritize security and sustainability. | 0.23 |  |
| Keywords | generative AI, cloud computing, cybersecurity, RAG, multi-cloud, Kubernetes, AI adoption, market share, zero trust, sustainability | None |  |
| Recommendations | Invest in RAG-based Al architectures for knowledge management applications., Establish robust document parsing pipelines to feed Al systems with high-quality data., Adopt a multi-cloud strategy with consistent governance and security policies., Prioritize cybersecurity investments, especially in zero-trust and supply chain security., Build internal Al literacy through training programs and centers of excellence., Evaluate sustainability metrics for all technology decisions and vendor selections. | None |  |

**Sections with a classified topic.** This is "segmentation by topic" through the schema: every heading gets a topic from a fixed list, which becomes a search filter in Part 6.

In [15]:
sections_rows = array_rows("Sections")
show_table(sections_rows, ["Heading", "Topic", "StartPage", "Summary", "min_confidence"])

| Heading | Topic | StartPage | Summary | min_confidence |
| --- | --- | --- | --- | --- |
| Executive Summary | General | 1 | Introduces the report’s purpose and frames the major technology trends shaping industry decisions in 2024. | 0.313 |
| Key Findings | General | 1 | Summarizes the report’s headline statistics across AI adoption, cloud-native usage, cybersecurity spending, edge computing, sustainability, low-code/no-code, and API-first adoption. | 0.313 |
| Generative AI in the Enterprise | GenerativeAI | 1 | Describes how generative AI, LLMs, and RAG are being adopted in enterprises and why data infrastructure is critical to AI quality. | 0.313 |
| AI Adoption by Industry | GenerativeAI | 1 | Presents a table of AI adoption rates, primary use cases, and ROI across six industries. | 0.313 |
| Cloud and Infrastructure Trends | CloudInfrastructure | 1 | Explains the shift toward multi-cloud and hybrid-cloud strategies, Kubernetes standardization, and infrastructure priorities for 2024. | 0.313 |
| Top Cloud Priorities for 2024 | CloudInfrastructure | 1 | Lists the main operational and strategic priorities for cloud teams, including cost, governance, serverless, IaC, and green computing. | 0.313 |
| Cloud Provider Market Share | CloudInfrastructure | 1 | Shows market share, growth, and strengths for major cloud providers across two table segments. | 0.313 |
| Cybersecurity Landscape | Cybersecurity | 1 | Describes rising ransomware, AI-powered threats, zero-trust adoption, and growing supply chain security concerns. | 0.313 |
| Recommendations | Recommendations | 1 | Recommends actions to improve AI capability, cloud governance, cybersecurity, internal literacy, and sustainability practices. | 0.313 |

**Tables as typed rows.** Check `CloudProviderMarketShare`: all four providers in **one** list even though the printed table is split across pages 3 and 4, and percentages are numbers ready for analytics.

In [16]:
for name in ["IndustryAIAdoption", "CloudProviderMarketShare"]:
    display(Markdown(f"#### {name}"))
    show_table(array_rows(name))

#### IndustryAIAdoption

| Industry | AdoptionRate | PrimaryUseCase | AvgROI | min_confidence | pages |
| --- | --- | --- | --- | --- | --- |
| Financial Services | 89.0 | Fraud Detection & Risk Analysis | 340.0 | 0.532 | 2 |
| Healthcare | 76.0 | Diagnostic Assistance | 280.0 | 0.625 | 2 |
| Manufacturing | 72.0 | Predictive Maintenance | 310.0 | 0.649 | 2 |
| Retail | 81.0 | Personalization & Recommendations | 250.0 | 0.608 | 2 |
| Technology | 94.0 | Code Generation & Testing | 420.0 | 0.536 | 2 |
| Education | 58.0 | Personalized Learning | 180.0 | 0.635 | 2 |

#### CloudProviderMarketShare

| Provider | MarketShare | YoYGrowth | KeyStrength | min_confidence | pages |
| --- | --- | --- | --- | --- | --- |
| AWS | 31.0 | 12.0 | Breadth of services | 0.609 | 2 |
| Microsoft Azure | 25.0 | 18.0 | Enterprise integration | 0.615 | 2 |
| Google Cloud | 11.0 | 22.0 | AI/ML capabilities | 0.609 | 3 |
| Others | 33.0 | 8.0 | Specialized offerings | 0.614 | 3 |

### 3.5 Grounding
Each value's `source` points back to the page and region it came from (`D(page, x1, y1, ...)`). A UI can highlight it; a workflow can send low-confidence values to human review.

In [17]:
for name in ["Title", "DocumentType"]:
    field = fields.get(name)
    if field:
        print(f"{name}: {to_plain(field)!r} | confidence={field.confidence} | source={field.source}")

first_row = (fields["CloudProviderMarketShare"].value_array or [None])[0] if "CloudProviderMarketShare" in fields else None
if first_row:
    for key, sub in (first_row.value_object or {}).items():
        print(f"CloudProviderMarketShare[0].{key}: {to_plain(sub)!r} | confidence={sub.confidence} | source={sub.source}")

Title: '2024 Technology Trends Report' | confidence=0.835 | source=D(1,1.9102,1.1114,6.5883,1.1330,6.5867,1.4853,1.9086,1.4637)
DocumentType: 'IndustryReport' | confidence=0.835 | source=D(1,1.9102,1.1114,6.5883,1.1330,6.5867,1.4853,1.9086,1.4637)
CloudProviderMarketShare[0].Provider: 'AWS' | confidence=0.864 | source=D(2,2.3346,9.1372,2.6646,9.1354,2.6646,9.2748,2.3346,9.2820)
CloudProviderMarketShare[0].MarketShare: 31.0 | confidence=0.609 | source=D(2,3.4427,9.1355,3.7297,9.1408,3.7270,9.2865,3.4400,9.2811)
CloudProviderMarketShare[0].YoYGrowth: 12.0 | confidence=0.817 | source=D(2,4.4036,9.1416,4.7688,9.1416,4.7688,9.2812,4.4036,9.2812)
CloudProviderMarketShare[0].KeyStrength: 'Breadth of services' | confidence=0.684 | source=D(2,5.2046,9.1362,6.4124,9.1362,6.4124,9.2866,5.2046,9.2866)


## 4. From CU output to search records

From here on everything uses `doc` (the custom analyzer result). It has the same Markdown and structure as `prebuilt-layout`, plus the fields.

### 4.1 Offset → page lookup
Each page lists the character spans it covers in `doc.markdown`, so any chunk's offsets map to exact page numbers.

In [18]:
markdown = doc.markdown
page_ranges = [(span.offset, span.offset + span.length, page.page_number) for page in doc.pages for span in page.spans or []]


def pages_for(start: int, end: int) -> tuple[int | None, int | None]:
    hits = [page for s, e, page in page_ranges if s < end and start < e]
    return (min(hits), max(hits)) if hits else (None, None)


print(pages_for(0, 50), pages_for(len(markdown) - 50, len(markdown)))

(1, 1) (3, 3)


### 4.2 Merge tables split across pages
Two consecutive tables are merged when they have the same column count and only page markers (`<!-- ... -->`) between them. A repeated header row is dropped. If CU already returned one table, this step changes nothing.

In [19]:
MARKER_RE = re.compile(r"<!--.*?-->", re.S)


def normalize(text: str) -> str:
    return re.sub(r"[^a-z0-9]", "", text.lower())


table_blocks = []
for table in sorted(doc.tables or [], key=lambda t: t.span.offset):
    start, end = table.span.offset, table.span.offset + table.span.length
    block = {"start": start, "end": end, "text": markdown[start:end].strip(), "columns": table.column_count, "parts": 1}
    prev = table_blocks[-1] if table_blocks else None
    if prev and prev["columns"] == block["columns"] and not MARKER_RE.sub("", markdown[prev["end"]:start]).strip():
        prev_lines, new_lines = prev["text"].splitlines(), block["text"].splitlines()
        if new_lines and normalize(new_lines[0]) == normalize(prev_lines[0]):
            new_lines = new_lines[2:]  # repeated header + separator row
        prev.update(text="\n".join(prev_lines + new_lines), end=end, parts=prev["parts"] + 1)
    else:
        table_blocks.append(block)

for block in table_blocks:
    block["pages"] = pages_for(block["start"], block["end"])
    print(f"--- table on pages {block['pages']} (merged from {block['parts']} part(s)) ---\n{block['text']}\n")

--- table on pages (2, 2) (merged from 1 part(s)) ---
| Industry | Adoption Rate | Primary Use Case | Avg. ROI |
| - | - | - | - |
| Financial Services | 89% | Fraud Detection & Risk Analysis | 340% |
| Healthcare | 76% | Diagnostic Assistance | 280% |
| Manufacturing | 72% | Predictive Maintenance | 310% |
| Retail | 81% | Personalization & Recommendations | 250% |
| Technology | 94% | Code Generation & Testing | 420% |
| Education | 58% | Personalized Learning | 180% |

--- table on pages (2, 3) (merged from 1 part(s)) ---
| Provider | Market Share | YoY Growth | Key Strength |
| - | - | - | - |
| AWS | 31% | +12% | Breadth of services |
| Microsoft Azure | 25% | +18% | Enterprise integration |
| Provider | Market Share | YoY Growth | Key Strength |
| Google Cloud | 11% | +22% | AI/ML capabilities |
| Others | 33% | +8% | Specialized offerings |



### 4.3 Section-aware chunking
1. Split the Markdown at headings, keeping the heading path (H1 > H2 > H3) and character offsets.
2. Inside a section, tables become **atomic** chunks (never split).
3. Text between tables is split by size, then cleaned: page markers removed and a paragraph broken by a page break is re-joined (previous text has no closing punctuation and the next starts lowercase).
4. Page numbers come from the **raw** offsets, before cleaning.

In [20]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

HEADING_RE = re.compile(r"^(#{1,6})\s+(.+?)\s*$", re.M)
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1200, chunk_overlap=150, add_start_index=True)


def clean_text(text: str) -> str:
    text = MARKER_RE.sub("", text)
    text = re.sub(r"(?<![.!?:;])\n\s*\n\s*(?=[a-z])", " ", text)
    return re.sub(r"\n{3,}", "\n\n", text).strip()


# (body_start, body_end, heading_path)
headings = list(HEADING_RE.finditer(markdown))
starts = [m.start() for m in headings] + [len(markdown)]
sections = [(0, starts[0], [])]
stack = []
for match, next_start in zip(headings, starts[1:]):
    level, title = len(match.group(1)), match.group(2).strip()
    stack = [(lvl, t) for lvl, t in stack if lvl < level] + [(level, title)]
    sections.append((match.end(), next_start, [t for _, t in stack]))

records = []
for body_start, body_end, path in sections:
    tables_here = [b for b in table_blocks if body_start <= b["start"] < body_end]
    cursor, pieces = body_start, []
    for block in tables_here:
        pieces.append(("text", cursor, block["start"]))
        pieces.append(("table", block["start"], block["end"]))
        cursor = max(cursor, block["end"])
    pieces.append(("text", cursor, body_end))

    for kind, start, end in pieces:
        if kind == "table":
            block = next(b for b in tables_here if b["start"] == start)
            records.append({"chunk_type": "table", "text": block["text"], "headers": path, "pages": block["pages"]})
            continue
        for piece in text_splitter.create_documents([markdown[start:end]]):
            text = clean_text(piece.page_content)
            if text:
                piece_start = start + piece.metadata["start_index"]
                records.append({"chunk_type": "text", "text": text, "headers": path, "pages": pages_for(piece_start, piece_start + len(piece.page_content))})

show_table([{"type": r["chunk_type"], "path": " > ".join(r["headers"]), "pages": r["pages"], "chars": len(r["text"]), "start": r["text"][:70]} for r in records])

| type | path | pages | chars | start |
| --- | --- | --- | --- | --- |
| text | 2024 Technology Trends Report > Executive Summary | 1, 1 | 342 | This report provides an analysis of the most significant technology tr |
| text | 2024 Technology Trends Report > Key Findings | 1, 1 | 637 | . Generative Al adoption has increased by 340% year-over-year across e |
| text | 2024 Technology Trends Report > Generative AI in the Enterprise | 1, 2 | 781 | Generative AI has emerged as the defining technology of 2024. Large la |
| table | 2024 Technology Trends Report > Generative AI in the Enterprise > AI Adoption by Industry | 2, 2 | 421 | \| Industry \| Adoption Rate \| Primary Use Case \| Avg. ROI \| \| - \| - \| - |
| text | 2024 Technology Trends Report > Cloud and Infrastructure Trends | 2, 2 | 328 | The cloud computing landscape continues to evolve rapidly. Multi-cloud |
| text | 2024 Technology Trends Report > Cloud and Infrastructure Trends > Top Cloud Priorities for 2024 | 2, 2 | 369 | . Cost optimization and FinOps practices to manage growing cloud spend |
| table | 2024 Technology Trends Report > Cloud and Infrastructure Trends > Cloud Provider Market Share | 2, 3 | 328 | \| Provider \| Market Share \| YoY Growth \| Key Strength \| \| - \| - \| - \|  |
| text | 2024 Technology Trends Report > Cybersecurity Landscape | 3, 3 | 715 | The cybersecurity threat landscape has become increasingly complex. Ra |
| text | 2024 Technology Trends Report > Recommendations | 3, 3 | 512 | . Invest in RAG-based Al architectures for knowledge management applic |

### 4.4 Extra records from CU fields
- **summary**: one record with the CU summary and keywords — good for "what is this document about?" questions.
- **table_row**: one record per extracted table row, written as a sentence and also kept as JSON (`row_json`) for downstream workflows. Short, precise records rank well for questions about a single value.

`TABLE_FIELDS` links each table field to the heading it belongs to, so rows inherit the heading path.

In [21]:
TABLE_FIELDS = {"IndustryAIAdoption": "AI Adoption by Industry", "CloudProviderMarketShare": "Cloud Provider Market Share"}

doc_title = to_plain(fields.get("Title")) or (headings[0].group(2).strip() if headings else DOC_PATH.stem)
doc_type = to_plain(fields.get("DocumentType")) or "Other"
keywords = to_plain(fields.get("Keywords")) or []

records = [r for r in records if r["chunk_type"] in ("text", "table")]  # makes the cell re-runnable
records.append(
    {
        "chunk_type": "summary",
        "text": f"{to_plain(fields.get('Summary')) or ''}\n\nKeywords: {', '.join(keywords)}",
        "headers": ["Document summary"],
        "pages": (doc.start_page_number, doc.end_page_number),
    }
)

for field_name, heading in TABLE_FIELDS.items():
    path = next((r["headers"] for r in records if r["chunk_type"] == "table" and r["headers"] and normalize(r["headers"][-1]) == normalize(heading)), [heading])
    for row in array_rows(field_name):
        values = {k: v for k, v in row.items() if k not in ("min_confidence", "pages")}
        pages = row["pages"] or [None]
        records.append(
            {
                "chunk_type": "table_row",
                "text": f"{heading} row: " + ", ".join(f"{k}: {v}" for k, v in values.items()),
                "headers": path,
                "pages": (min(pages), max(pages)),
                "row_json": json.dumps(values),
            }
        )

print(f"{len(records)} records:", {t: sum(r['chunk_type'] == t for r in records) for t in ('text', 'table', 'table_row', 'summary')})

20 records: {'text': 7, 'table': 2, 'table_row': 10, 'summary': 1}


### 4.5 Final search documents
- **Topic** comes from the CU `Sections` field: the deepest heading in the path that CU classified.
- **Contextual header**: `content` starts with `title | section path | pages` so keyword and vector search (and the LLM) know where a chunk comes from — especially important for bare tables.

In [22]:
section_topics = {normalize(row["Heading"]): row["Topic"] for row in sections_rows if row.get("Heading") and row.get("Topic")}


def topic_for(path: list[str]) -> str:
    return next((section_topics[normalize(h)] for h in reversed(path) if normalize(h) in section_topics), "General")


def fmt_pages(pages: tuple) -> str:
    start, end = pages
    return f"page {start}" if start == end else f"pages {start}-{end}"


documents = []
for i, r in enumerate(records):
    path = r["headers"]
    section_path = " > ".join(path) or doc_title
    documents.append(
        {
            "id": f"{DOC_PATH.stem}-{i:03d}",
            "content": f"{doc_title} | {section_path} | {fmt_pages(r['pages'])}\n\n{r['text']}",
            "chunk_type": r["chunk_type"],
            "section_path": section_path,
            "header_1": path[0] if len(path) > 0 else None,
            "header_2": path[1] if len(path) > 1 else None,
            "header_3": path[2] if len(path) > 2 else None,
            "page_start": r["pages"][0],
            "page_end": r["pages"][1],
            "topic": "General" if r["chunk_type"] == "summary" else topic_for(path),
            "doc_title": doc_title,
            "doc_type": doc_type,
            "keywords": keywords,
            "source_file": DOC_PATH.name,
            "row_json": r.get("row_json"),
        }
    )

show_table(documents, ["id", "chunk_type", "topic", "section_path", "page_start", "page_end"])
print(documents[0]["content"][:600])

| id | chunk_type | topic | section_path | page_start | page_end |
| --- | --- | --- | --- | --- | --- |
| mixed_content-000 | text | General | 2024 Technology Trends Report > Executive Summary | 1 | 1 |
| mixed_content-001 | text | General | 2024 Technology Trends Report > Key Findings | 1 | 1 |
| mixed_content-002 | text | GenerativeAI | 2024 Technology Trends Report > Generative AI in the Enterprise | 1 | 2 |
| mixed_content-003 | table | GenerativeAI | 2024 Technology Trends Report > Generative AI in the Enterprise > AI Adoption by Industry | 2 | 2 |
| mixed_content-004 | text | CloudInfrastructure | 2024 Technology Trends Report > Cloud and Infrastructure Trends | 2 | 2 |
| mixed_content-005 | text | CloudInfrastructure | 2024 Technology Trends Report > Cloud and Infrastructure Trends > Top Cloud Priorities for 2024 | 2 | 2 |
| mixed_content-006 | table | CloudInfrastructure | 2024 Technology Trends Report > Cloud and Infrastructure Trends > Cloud Provider Market Share | 2 | 3 |
| mixed_content-007 | text | Cybersecurity | 2024 Technology Trends Report > Cybersecurity Landscape | 3 | 3 |
| mixed_content-008 | text | Recommendations | 2024 Technology Trends Report > Recommendations | 3 | 3 |
| mixed_content-009 | summary | General | Document summary | 1 | 3 |
| mixed_content-010 | table_row | GenerativeAI | 2024 Technology Trends Report > Generative AI in the Enterprise > AI Adoption by Industry | 2 | 2 |
| mixed_content-011 | table_row | GenerativeAI | 2024 Technology Trends Report > Generative AI in the Enterprise > AI Adoption by Industry | 2 | 2 |
| mixed_content-012 | table_row | GenerativeAI | 2024 Technology Trends Report > Generative AI in the Enterprise > AI Adoption by Industry | 2 | 2 |
| mixed_content-013 | table_row | GenerativeAI | 2024 Technology Trends Report > Generative AI in the Enterprise > AI Adoption by Industry | 2 | 2 |
| mixed_content-014 | table_row | GenerativeAI | 2024 Technology Trends Report > Generative AI in the Enterprise > AI Adoption by Industry | 2 | 2 |
| mixed_content-015 | table_row | GenerativeAI | 2024 Technology Trends Report > Generative AI in the Enterprise > AI Adoption by Industry | 2 | 2 |
| mixed_content-016 | table_row | CloudInfrastructure | 2024 Technology Trends Report > Cloud and Infrastructure Trends > Cloud Provider Market Share | 2 | 2 |
| mixed_content-017 | table_row | CloudInfrastructure | 2024 Technology Trends Report > Cloud and Infrastructure Trends > Cloud Provider Market Share | 2 | 2 |
| mixed_content-018 | table_row | CloudInfrastructure | 2024 Technology Trends Report > Cloud and Infrastructure Trends > Cloud Provider Market Share | 3 | 3 |
| mixed_content-019 | table_row | CloudInfrastructure | 2024 Technology Trends Report > Cloud and Infrastructure Trends > Cloud Provider Market Share | 3 | 3 |

2024 Technology Trends Report | 2024 Technology Trends Report > Executive Summary | page 1

This report provides an analysis of the most significant technology trends shaping the industry
in 2024. From the rapid adoption of generative AI to advances in quantum computing, these
trends are reshaping how businesses operate and compete. Understanding these trends is
essential for strategic planning and technology investment decisions.


## 5. Azure AI Search index

| Field | Attributes | Purpose |
| --- | --- | --- |
| `id` | key | `<file>-<n>` |
| `content` | searchable | text the LLM reads (with contextual header) |
| `content_vector` | 3072-d, HNSW | vector search |
| `chunk_type` | filterable, facetable | `text` / `table` / `table_row` / `summary` |
| `section_path`, `header_1..3` | searchable, filterable | citations, filters |
| `page_start`, `page_end` | filterable, sortable | page references |
| `topic`, `doc_type` | filterable, facetable | CU classify fields |
| `keywords` | searchable, filterable, facetable | CU generate field |
| `doc_title`, `source_file` | searchable / filterable | provenance |
| `row_json` | retrievable only | structured row for workflows |

The semantic configuration is added only when `USE_SEMANTIC_RANKER` is on. Changing field definitions of an existing index requires deleting it first (see Part 10).

In [25]:
from azure.search.documents.indexes.models import (
    HnswAlgorithmConfiguration,
    SearchField,
    SearchIndex,
    SemanticConfiguration,
    SemanticField,
    SemanticPrioritizedFields,
    SemanticSearch,
    VectorSearch,
    VectorSearchProfile,
)

fields_def = [
    SearchField(name="id", type="Edm.String", key=True, filterable=True),
    SearchField(name="content", type="Edm.String", searchable=True),
    SearchField(name="content_vector", type="Collection(Edm.Single)", searchable=True,
                vector_search_dimensions=EMBEDDING_DIMENSIONS, vector_search_profile_name="hnsw-profile"),
    SearchField(name="chunk_type", type="Edm.String", filterable=True, facetable=True),
    SearchField(name="section_path", type="Edm.String", searchable=True, filterable=True),
    SearchField(name="header_1", type="Edm.String", searchable=True, filterable=True),
    SearchField(name="header_2", type="Edm.String", searchable=True, filterable=True),
    SearchField(name="header_3", type="Edm.String", searchable=True, filterable=True),
    SearchField(name="page_start", type="Edm.Int32", filterable=True, sortable=True),
    SearchField(name="page_end", type="Edm.Int32", filterable=True, sortable=True),
    SearchField(name="topic", type="Edm.String", filterable=True, facetable=True),
    SearchField(name="doc_title", type="Edm.String", searchable=True),
    SearchField(name="doc_type", type="Edm.String", filterable=True, facetable=True),
    SearchField(name="keywords", type="Collection(Edm.String)", searchable=True, filterable=True, facetable=True),
    SearchField(name="source_file", type="Edm.String", filterable=True),
    SearchField(name="row_json", type="Edm.String"),
]

semantic_search = SemanticSearch(
    default_configuration_name="semantic-config",
    configurations=[
        SemanticConfiguration(
            name="semantic-config",
            prioritized_fields=SemanticPrioritizedFields(
                title_field=SemanticField(field_name="section_path"),
                content_fields=[SemanticField(field_name="content")],
                keywords_fields=[SemanticField(field_name="topic")],
            ),
        )
    ],
)

index = SearchIndex(
    name=INDEX_NAME,
    fields=fields_def,
    vector_search=VectorSearch(
        algorithms=[HnswAlgorithmConfiguration(name="hnsw-config")],
        profiles=[VectorSearchProfile(name="hnsw-profile", algorithm_configuration_name="hnsw-config")],
    ),
    semantic_search=semantic_search if USE_SEMANTIC_RANKER else None,
)
index_client.create_or_update_index(index)
print(f"Index '{INDEX_NAME}' is ready")

Index 'mixed-content-rag' is ready


### 5.1 Embed and upload
`embed()` also counts embedding tokens (index vs. query) for the cost section. Old records of this file are deleted first, so re-runs with fewer chunks leave nothing stale behind.

In [26]:
TOKEN_LOG = {"index_embedding_tokens": 0, "query_embedding_tokens": 0}


def embed(texts: list[str], purpose: str = "query") -> list[list[float]]:
    response = openai_client.embeddings.create(model=EMBEDDING_DEPLOYMENT, input=texts, dimensions=EMBEDDING_DIMENSIONS)
    TOKEN_LOG[f"{purpose}_embedding_tokens"] += response.usage.total_tokens
    return [item.embedding for item in response.data]


stale = [{"id": d["id"]} for d in search_client.search(search_text="*", filter=f"source_file eq '{DOC_PATH.name}'", select=["id"], top=1000)]
if stale:
    search_client.delete_documents(documents=stale)

BATCH_SIZE = 16
for start in range(0, len(documents), BATCH_SIZE):
    batch = documents[start : start + BATCH_SIZE]
    for document, vector in zip(batch, embed([d["content"] for d in batch], purpose="index")):
        document["content_vector"] = vector

upload_results = search_client.merge_or_upload_documents(documents=documents)
print(f"Deleted {len(stale)} stale, uploaded {sum(r.succeeded for r in upload_results)}/{len(documents)} | "
      f"embedding tokens: {TOKEN_LOG['index_embedding_tokens']}")

Deleted 0 stale, uploaded 20/20 | embedding tokens: 1831


In [27]:
# The count can lag a few seconds behind the upload.
print("Documents in index:", search_client.get_document_count())
facets = search_client.search(search_text="*", facets=["chunk_type", "topic"], top=0).get_facets()
print(json.dumps(facets, indent=2))

Documents in index: 20
{
  "chunk_type": [
    {
      "value": "table_row",
      "count": 10
    },
    {
      "value": "text",
      "count": 7
    },
    {
      "value": "table",
      "count": 2
    },
    {
      "value": "summary",
      "count": 1
    }
  ],
  "topic": [
    {
      "value": "GenerativeAI",
      "count": 8
    },
    {
      "value": "CloudInfrastructure",
      "count": 7
    },
    {
      "value": "General",
      "count": 3
    },
    {
      "value": "Cybersecurity",
      "count": 1
    },
    {
      "value": "Recommendations",
      "count": 1
    }
  ]
}


## 6. Retrieval

| Mode | What it does |
| --- | --- |
| `vector` | nearest neighbours on `content_vector` |
| `hybrid` | BM25 keyword + vector, fused with Reciprocal Rank Fusion |
| `semantic` | hybrid, then re-ranked by the semantic ranker (counts against the free monthly allowance) |

`filter` takes an OData expression, e.g. `chunk_type eq 'table'`.

In [28]:
from azure.search.documents.models import VectorizedQuery

SELECT_FIELDS = ["id", "content", "chunk_type", "section_path", "page_start", "page_end", "topic", "row_json"]


def retrieve(query: str, k: int = 5, mode: str = "semantic", filter: str | None = None) -> list[dict]:
    if mode == "semantic" and not USE_SEMANTIC_RANKER:
        mode = "hybrid"
    vector_query = VectorizedQuery(vector=embed([query])[0], k_nearest_neighbors=k if mode == "vector" else 50, fields="content_vector")
    params = {
        "search_text": None if mode == "vector" else query,
        "vector_queries": [vector_query],
        "select": SELECT_FIELDS,
        "filter": filter,
        "top": k,
    }
    if mode == "semantic":
        params.update(query_type="semantic", semantic_configuration_name="semantic-config")
    return list(search_client.search(**params))


def show_hits(hits: list[dict]) -> None:
    show_table(
        [
            {
                "rank": i,
                "score": round(h.get("@search.reranker_score") or h["@search.score"], 3),
                "type": h["chunk_type"],
                "topic": h["topic"],
                "section": h["section_path"],
                "pages": f"{h['page_start']}-{h['page_end']}",
            }
            for i, h in enumerate(hits, 1)
        ]
    )

### 6.1 Compare modes on a text, a table and a cross-page table question

In [29]:
PROBE_QUESTIONS = [
    "Why has RAG become the preferred architecture for enterprise AI?",
    "What is the AI adoption rate in healthcare?",
    "What is Google Cloud's market share and year-over-year growth?",
]
modes = ["vector", "hybrid"] + (["semantic"] if USE_SEMANTIC_RANKER else [])

for question in PROBE_QUESTIONS:
    for mode in modes:
        display(Markdown(f"**{question}** — `{mode}`"))
        show_hits(retrieve(question, k=3, mode=mode))

**Why has RAG become the preferred architecture for enterprise AI?** — `vector`

| rank | score | type | topic | section | pages |
| --- | --- | --- | --- | --- | --- |
| 1 | 0.683 | text | GenerativeAI | 2024 Technology Trends Report > Generative AI in the Enterprise | 1-2 |
| 2 | 0.642 | text | Recommendations | 2024 Technology Trends Report > Recommendations | 3-3 |
| 3 | 0.628 | summary | General | Document summary | 1-3 |

**Why has RAG become the preferred architecture for enterprise AI?** — `hybrid`

| rank | score | type | topic | section | pages |
| --- | --- | --- | --- | --- | --- |
| 1 | 0.033 | text | GenerativeAI | 2024 Technology Trends Report > Generative AI in the Enterprise | 1-2 |
| 2 | 0.031 | table_row | GenerativeAI | 2024 Technology Trends Report > Generative AI in the Enterprise > AI Adoption by Industry | 2-2 |
| 3 | 0.031 | table_row | GenerativeAI | 2024 Technology Trends Report > Generative AI in the Enterprise > AI Adoption by Industry | 2-2 |

**Why has RAG become the preferred architecture for enterprise AI?** — `semantic`

| rank | score | type | topic | section | pages |
| --- | --- | --- | --- | --- | --- |
| 1 | 3.253 | text | GenerativeAI | 2024 Technology Trends Report > Generative AI in the Enterprise | 1-2 |
| 2 | 2.172 | text | Recommendations | 2024 Technology Trends Report > Recommendations | 3-3 |
| 3 | 2.078 | summary | General | Document summary | 1-3 |

... (8 outputs left out)

**What is Google Cloud's market share and year-over-year growth?** — `hybrid`

| rank | score | type | topic | section | pages |
| --- | --- | --- | --- | --- | --- |
| 1 | 0.033 | table | CloudInfrastructure | 2024 Technology Trends Report > Cloud and Infrastructure Trends > Cloud Provider Market Share | 2-3 |
| 2 | 0.033 | table_row | CloudInfrastructure | 2024 Technology Trends Report > Cloud and Infrastructure Trends > Cloud Provider Market Share | 3-3 |
| 3 | 0.032 | table_row | CloudInfrastructure | 2024 Technology Trends Report > Cloud and Infrastructure Trends > Cloud Provider Market Share | 2-2 |

**What is Google Cloud's market share and year-over-year growth?** — `semantic`

| rank | score | type | topic | section | pages |
| --- | --- | --- | --- | --- | --- |
| 1 | 2.694 | table_row | CloudInfrastructure | 2024 Technology Trends Report > Cloud and Infrastructure Trends > Cloud Provider Market Share | 3-3 |
| 2 | 2.65 | table | CloudInfrastructure | 2024 Technology Trends Report > Cloud and Infrastructure Trends > Cloud Provider Market Share | 2-3 |
| 3 | 2.162 | table_row | CloudInfrastructure | 2024 Technology Trends Report > Cloud and Infrastructure Trends > Cloud Provider Market Share | 3-3 |

### 6.2 Metadata filters
The CU-derived metadata narrows the search space before ranking.

In [30]:
show_hits(retrieve("market share", k=5, filter="chunk_type eq 'table_row'"))
show_hits(retrieve("main threats", k=3, filter="topic eq 'Cybersecurity'"))

| rank | score | type | topic | section | pages |
| --- | --- | --- | --- | --- | --- |
| 1 | 2.726 | table_row | CloudInfrastructure | 2024 Technology Trends Report > Cloud and Infrastructure Trends > Cloud Provider Market Share | 2-2 |
| 2 | 2.704 | table_row | CloudInfrastructure | 2024 Technology Trends Report > Cloud and Infrastructure Trends > Cloud Provider Market Share | 3-3 |
| 3 | 2.691 | table_row | CloudInfrastructure | 2024 Technology Trends Report > Cloud and Infrastructure Trends > Cloud Provider Market Share | 2-2 |
| 4 | 2.652 | table_row | CloudInfrastructure | 2024 Technology Trends Report > Cloud and Infrastructure Trends > Cloud Provider Market Share | 3-3 |
| 5 | 1.738 | table_row | GenerativeAI | 2024 Technology Trends Report > Generative AI in the Enterprise > AI Adoption by Industry | 2-2 |

| rank | score | type | topic | section | pages |
| --- | --- | --- | --- | --- | --- |
| 1 | 2.545 | text | Cybersecurity | 2024 Technology Trends Report > Cybersecurity Landscape | 3-3 |

## 7. Grounded answers with citations

Each retrieved record is numbered `[n]`. Its `content` already starts with `title | section | pages`, so the model can cite and the reader can verify. All calls use the Responses API and return token usage and latency for Part 9.

In [31]:
ANSWER_INSTRUCTIONS = (
    "You answer questions about a document using ONLY the numbered sources provided.\n"
    "- Cite every fact with its source number in square brackets, e.g. [2].\n"
    "- Tables are markdown; read numbers exactly as written.\n"
    "- If the sources do not contain the answer, reply exactly: The document does not say.\n"
    "- Be concise."
)


def ask_llm(model: str, instructions: str, user_input) -> tuple:
    extra = {"reasoning": {"effort": REASONING_EFFORT}} if REASONING_EFFORT else {}
    start = time.perf_counter()
    response = openai_client.responses.create(model=model, instructions=instructions, input=user_input, **extra)
    return response, time.perf_counter() - start


def usage_of(response) -> dict:
    usage = response.usage
    details = getattr(usage, "output_tokens_details", None)
    return {
        "input_tokens": usage.input_tokens,
        "output_tokens": usage.output_tokens,  # includes reasoning tokens
        "reasoning_tokens": getattr(details, "reasoning_tokens", 0) or 0,
    }


def answer_with_rag(question: str, k: int = 5, mode: str = "semantic") -> dict:
    start = time.perf_counter()
    hits = retrieve(question, k=k, mode=mode)
    sources = "\n\n".join(f"[{i}] {h['content']}" for i, h in enumerate(hits, 1))
    response, _ = ask_llm(CHAT_DEPLOYMENT, ANSWER_INSTRUCTIONS, f"Sources:\n{sources}\n\nQuestion: {question}")
    return {"answer": response.output_text, "sources": hits, **usage_of(response), "seconds": round(time.perf_counter() - start, 2)}


def show_answer(result: dict) -> None:
    refs = "\n".join(
        f"- [{i}] {h['section_path']} — pages {h['page_start']}-{h['page_end']} ({h['chunk_type']})"
        for i, h in enumerate(result["sources"], 1)
    )
    stats = f"{result['input_tokens']} in / {result['output_tokens']} out tokens, {result['seconds']}s"
    display(Markdown(f"**Answer** ({stats})\n\n{result['answer']}\n\n**References**\n{refs}"))

In [32]:
for question in [
    "Which industry has the highest average ROI from AI, and what is its main use case?",
    "Which cloud provider grew fastest year over year?",
    "What is Oracle Cloud's market share?",  # not in the document
]:
    display(Markdown(f"### {question}"))
    show_answer(answer_with_rag(question))

### Which industry has the highest average ROI from AI, and what is its main use case?

**Answer** (516 in / 60 out tokens, 6.51s)

Technology has the highest average ROI from AI at 420%, and its main use case is Code Generation & Testing. [4]

**References**
- [1] 2024 Technology Trends Report > Generative AI in the Enterprise > AI Adoption by Industry — pages 2-2 (table_row)
- [2] 2024 Technology Trends Report > Generative AI in the Enterprise > AI Adoption by Industry — pages 2-2 (table_row)
- [3] 2024 Technology Trends Report > Generative AI in the Enterprise > AI Adoption by Industry — pages 2-2 (table_row)
- [4] 2024 Technology Trends Report > Generative AI in the Enterprise > AI Adoption by Industry — pages 2-2 (table_row)
- [5] 2024 Technology Trends Report > Generative AI in the Enterprise > AI Adoption by Industry — pages 2-2 (table)

### Which cloud provider grew fastest year over year?

**Answer** (566 in / 49 out tokens, 4.02s)

Google Cloud grew fastest year over year at 22%.[1][3]

**References**
- [1] 2024 Technology Trends Report > Cloud and Infrastructure Trends > Cloud Provider Market Share — pages 2-3 (table)
- [2] 2024 Technology Trends Report > Cloud and Infrastructure Trends > Cloud Provider Market Share — pages 2-2 (table_row)
- [3] 2024 Technology Trends Report > Cloud and Infrastructure Trends > Cloud Provider Market Share — pages 3-3 (table_row)
- [4] 2024 Technology Trends Report > Cloud and Infrastructure Trends > Cloud Provider Market Share — pages 2-2 (table_row)
- [5] 2024 Technology Trends Report > Key Findings — pages 1-1 (text)

### What is Oracle Cloud's market share?

**Answer** (546 in / 32 out tokens, 4.18s)

The document does not say.

**References**
- [1] 2024 Technology Trends Report > Cloud and Infrastructure Trends > Cloud Provider Market Share — pages 2-2 (table_row)
- [2] 2024 Technology Trends Report > Cloud and Infrastructure Trends > Cloud Provider Market Share — pages 3-3 (table_row)
- [3] 2024 Technology Trends Report > Cloud and Infrastructure Trends > Cloud Provider Market Share — pages 3-3 (table_row)
- [4] Document summary — pages 1-3 (summary)
- [5] 2024 Technology Trends Report > Cloud and Infrastructure Trends > Cloud Provider Market Share — pages 2-3 (table)

## 8. Agent: the model decides what to search

The model gets a `search_document` tool and calls it as often as needed (up to `max_rounds`), choosing filters itself — e.g. `chunk_type='table_row'` for numbers. Strict JSON schema + `enum` values keep the generated filters safe. `previous_response_id` chains the turns.

In [33]:
SEARCH_TOOL = {
    "type": "function",
    "name": "search_document",
    "description": "Search the indexed report. Returns numbered passages with section path and pages. "
                   "Use chunk_type 'table_row' or 'table' for numbers from tables.",
    "parameters": {
        "type": "object",
        "properties": {
            "query": {"type": "string", "description": "Focused search query."},
            "chunk_type": {"type": ["string", "null"], "enum": ["text", "table", "table_row", "summary", None]},
            "topic": {"type": ["string", "null"], "enum": [*TOPICS, None]},
        },
        "required": ["query", "chunk_type", "topic"],
        "additionalProperties": False,
    },
    "strict": True,
}

AGENT_INSTRUCTIONS = (
    "You are a research assistant for an indexed document. Always call search_document before answering; "
    "call it several times for multi-part questions. Answer only from tool results and cite passages as [n] "
    "using the numbers in the tool output. If nothing relevant is found, reply exactly: The document does not say."
)


def run_agent(question: str, max_rounds: int = 4) -> dict:
    extra = {"reasoning": {"effort": REASONING_EFFORT}} if REASONING_EFFORT else {}
    totals = {"input_tokens": 0, "output_tokens": 0, "reasoning_tokens": 0}
    sources, next_input, previous_id = [], [{"role": "user", "content": question}], None
    start = time.perf_counter()
    for _ in range(max_rounds):
        response = openai_client.responses.create(
            model=CHAT_DEPLOYMENT, instructions=AGENT_INSTRUCTIONS, input=next_input,
            tools=[SEARCH_TOOL], previous_response_id=previous_id, **extra,
        )
        for key, value in usage_of(response).items():
            totals[key] += value
        calls = [item for item in response.output if item.type == "function_call"]
        if not calls:
            break
        previous_id, next_input = response.id, []
        for call in calls:
            args = json.loads(call.arguments)
            filters = " and ".join(f"{key} eq '{args[key]}'" for key in ("chunk_type", "topic") if args.get(key))
            hits = retrieve(args["query"], k=4, filter=filters or None)
            numbered = []
            for hit in hits:
                sources.append(hit)
                numbered.append(f"[{len(sources)}] {hit['content']}")
            print(f"tool call: {args} -> {len(hits)} hits")
            next_input.append({"type": "function_call_output", "call_id": call.call_id, "output": "\n\n".join(numbered) or "No results."})
    return {"answer": response.output_text, "sources": sources, **totals, "seconds": round(time.perf_counter() - start, 2)}

In [34]:
show_answer(run_agent(
    "Compare the fastest-growing cloud provider's YoY growth with the industry that has the highest AI adoption rate. "
    "Also, what does the report recommend?"
))

tool call: {'query': 'fastest-growing cloud provider YoY growth cloud provider growth percentage', 'chunk_type': 'text', 'topic': 'CloudInfrastructure'} -> 2 hits
tool call: {'query': 'industry highest AI adoption rate adoption rate industry percentage', 'chunk_type': 'text', 'topic': 'GenerativeAI'} -> 1 hits
tool call: {'query': 'recommendations report recommends cloud AI adoption', 'chunk_type': 'text', 'topic': 'Recommendations'} -> 1 hits
tool call: {'query': 'AWS Azure Google Cloud YoY growth fastest growing provider percentage', 'chunk_type': 'text', 'topic': 'CloudInfrastructure'} -> 2 hits
tool call: {'query': 'cloud provider growth year over year fastest growing AWS Azure Google Cloud report', 'chunk_type': 'text', 'topic': 'CloudInfrastructure'} -> 2 hits
tool call: {'query': 'highest AI adoption rate industry healthcare finance manufacturing retail percentage report', 'chunk_type': 'text', 'topic': 'GenerativeAI'} -> 1 hits
tool call: {'query': 'adoption rate industry table

**Answer** (5003 in / 596 out tokens, 29.78s)

The fastest-growing cloud provider is Google Cloud, with 22% YoY growth [16]. The industry with the highest AI adoption rate is Technology, at 94% [18].

The report recommends:
- investing in RAG-based AI architectures,
- establishing robust document parsing pipelines,
- adopting a multi-cloud strategy with consistent governance and security,
- prioritizing cybersecurity investments, especially zero-trust and supply chain security,
- building internal AI literacy through training and centers of excellence,
- and evaluating sustainability metrics for technology and vendor decisions [4].

**References**
- [1] 2024 Technology Trends Report > Cloud and Infrastructure Trends — pages 2-2 (text)
- [2] 2024 Technology Trends Report > Cloud and Infrastructure Trends > Top Cloud Priorities for 2024 — pages 2-2 (text)
- [3] 2024 Technology Trends Report > Generative AI in the Enterprise — pages 1-2 (text)
- [4] 2024 Technology Trends Report > Recommendations — pages 3-3 (text)
- [5] 2024 Technology Trends Report > Cloud and Infrastructure Trends — pages 2-2 (text)
- [6] 2024 Technology Trends Report > Cloud and Infrastructure Trends > Top Cloud Priorities for 2024 — pages 2-2 (text)
- [7] 2024 Technology Trends Report > Cloud and Infrastructure Trends — pages 2-2 (text)
- [8] 2024 Technology Trends Report > Cloud and Infrastructure Trends > Top Cloud Priorities for 2024 — pages 2-2 (text)
- [9] 2024 Technology Trends Report > Generative AI in the Enterprise — pages 1-2 (text)
- [10] 2024 Technology Trends Report > Generative AI in the Enterprise > AI Adoption by Industry — pages 2-2 (table_row)
- [11] 2024 Technology Trends Report > Generative AI in the Enterprise > AI Adoption by Industry — pages 2-2 (table_row)
- [12] 2024 Technology Trends Report > Generative AI in the Enterprise > AI Adoption by Industry — pages 2-2 (table_row)
- [13] 2024 Technology Trends Report > Generative AI in the Enterprise > AI Adoption by Industry — pages 2-2 (table_row)
- [14] 2024 Technology Trends Report > Cloud and Infrastructure Trends > Cloud Provider Market Share — pages 2-2 (table_row)
- [15] 2024 Technology Trends Report > Cloud and Infrastructure Trends > Cloud Provider Market Share — pages 2-2 (table_row)
- [16] 2024 Technology Trends Report > Cloud and Infrastructure Trends > Cloud Provider Market Share — pages 3-3 (table_row)
- [17] 2024 Technology Trends Report > Cloud and Infrastructure Trends > Cloud Provider Market Share — pages 3-3 (table_row)
- [18] 2024 Technology Trends Report > Generative AI in the Enterprise > AI Adoption by Industry — pages 2-2 (table)

## 9. Measured comparison

| Approach | Context sent to the model | Model |
| --- | --- | --- |
| **A** | Whole PDF (text + an image of every page) on **every** question | `gpt-5.4` |
| **B** | Whole CU Markdown (no retrieval) | `gpt-5.4-mini` |
| **C** | CU + AI Search top-k records | `gpt-5.4-mini` |

All three use the same instructions style and reasoning effort. A question counts as correct when **all** expected keywords appear in the answer (simple and explainable; read the answers too).

### 9.1 Evaluation set

In [35]:
EVAL_SET = [
    {"question": "What is the AI adoption rate in healthcare and its primary use case?", "expected": ["76", "diagnostic"], "kind": "table"},
    {"question": "Which industry has the highest average ROI from AI?", "expected": ["technology", "420"], "kind": "table reasoning"},
    {"question": "What is Google Cloud's market share and YoY growth?", "expected": ["11", "22"], "kind": "cross-page table"},
    {"question": "Which cloud provider has the fastest year-over-year growth?", "expected": ["google"], "kind": "cross-page table reasoning"},
    {"question": "What are Microsoft Azure's market share and key strength?", "expected": ["25", "enterprise integration"], "kind": "table"},
    {"question": "What does the report say about Kubernetes?", "expected": ["container orchestration"], "kind": "text"},
    {"question": "Why has RAG become the preferred architecture for enterprise AI?", "expected": ["fluency", "accuracy"], "kind": "text"},
    {"question": "What does SBOM stand for and why is it mentioned?", "expected": ["software bill of materials", "supply chain"], "kind": "text"},
    {"question": "What is the first recommendation in the report?", "expected": ["rag", "knowledge management"], "kind": "list"},
    {"question": "What is Oracle Cloud's market share?", "expected": ["does not say"], "kind": "not in document"},
]

### 9.2 The three approaches
A sends the PDF as an `input_file` (base64). B strips page markers from the CU Markdown. C is `answer_with_rag()` from Part 7.

In [36]:
BASELINE_INSTRUCTIONS = (
    "You answer questions using ONLY the attached document. Read table numbers exactly as written. "
    "If the document does not contain the answer, reply exactly: The document does not say. Be concise."
)
pdf_data_uri = "data:application/pdf;base64," + base64.b64encode(DOC_PATH.read_bytes()).decode()
full_markdown = MARKER_RE.sub("", doc.markdown)


def ask_full_pdf(question: str) -> dict:
    content = [
        {"type": "input_file", "filename": DOC_PATH.name, "file_data": pdf_data_uri},
        {"type": "input_text", "text": question},
    ]
    response, seconds = ask_llm(PREMIUM_DEPLOYMENT, BASELINE_INSTRUCTIONS, [{"role": "user", "content": content}])
    return {"answer": response.output_text, **usage_of(response), "seconds": round(seconds, 2)}


def ask_full_markdown(question: str) -> dict:
    response, seconds = ask_llm(CHAT_DEPLOYMENT, BASELINE_INSTRUCTIONS, f"Document:\n{full_markdown}\n\nQuestion: {question}")
    return {"answer": response.output_text, **usage_of(response), "seconds": round(seconds, 2)}


APPROACHES = {
    "A": ("Full PDF -> premium", ask_full_pdf),
    "B": ("CU markdown -> mini", ask_full_markdown),
    "C": ("CU + Search RAG -> mini", answer_with_rag),
}

### 9.3 Run the evaluation
About 30 model calls. Results are saved to `data/output/eval_results.json` for later presentation.

In [37]:
eval_rows = []
for item in EVAL_SET:
    for key, (label, ask) in APPROACHES.items():
        out = ask(item["question"])
        eval_rows.append(
            {
                "approach": key,
                "kind": item["kind"],
                "question": item["question"],
                "correct": all(word in out["answer"].lower() for word in item["expected"]),
                "input_tokens": out["input_tokens"],
                "output_tokens": out["output_tokens"],
                "reasoning_tokens": out["reasoning_tokens"],
                "seconds": out["seconds"],
                "answer": out["answer"],
            }
        )
    print(f"done: {item['question']}")

(OUTPUT_DIR / "eval_results.json").write_text(json.dumps(eval_rows, indent=2), encoding="utf-8")

done: What is the AI adoption rate in healthcare and its primary use case?
done: Which industry has the highest average ROI from AI?
done: What is Google Cloud's market share and YoY growth?
done: Which cloud provider has the fastest year-over-year growth?
done: What are Microsoft Azure's market share and key strength?
done: What does the report say about Kubernetes?
done: Why has RAG become the preferred architecture for enterprise AI?
done: What does SBOM stand for and why is it mentioned?
done: What is the first recommendation in the report?
done: What is Oracle Cloud's market share?


10503

In [38]:
summary = {}
for key, (label, _) in APPROACHES.items():
    rows = [r for r in eval_rows if r["approach"] == key]
    summary[key] = {
        "approach": f"{key}: {label}",
        "correct": f"{sum(r['correct'] for r in rows)}/{len(rows)}",
        "avg_input_tokens": round(mean(r["input_tokens"] for r in rows)),
        "avg_output_tokens": round(mean(r["output_tokens"] for r in rows)),
        "avg_reasoning_tokens": round(mean(r["reasoning_tokens"] for r in rows)),
        "avg_seconds": round(mean(r["seconds"] for r in rows), 2),
    }
show_table(list(summary.values()))

| approach | correct | avg_input_tokens | avg_output_tokens | avg_reasoning_tokens | avg_seconds |
| --- | --- | --- | --- | --- | --- |
| A: Full PDF -> premium | 10/10 | 939 | 28 | 9 | 2.97 |
| B: CU markdown -> mini | 10/10 | 988 | 47 | 24 | 3.01 |
| C: CU + Search RAG -> mini | 10/10 | 590 | 52 | 24 | 5.22 |

In [39]:
show_table(eval_rows, ["approach", "kind", "question", "correct", "input_tokens", "seconds", "answer"])

| approach | kind | question | correct | input_tokens | seconds | answer |
| --- | --- | --- | --- | --- | --- | --- |
| A | table | What is the AI adoption rate in healthcare and its primary use case? | True | 942 | 3.34 | Healthcare: 76% adoption rate; Diagnostic Assistance. |
| B | table | What is the AI adoption rate in healthcare and its primary use case? | True | 991 | 4.32 | Healthcare has an AI adoption rate of 76%, and its primary use case is Diagnostic Assistance. |
| C | table | What is the AI adoption rate in healthcare and its primary use case? | True | 510 | 4.62 | Healthcare has an AI adoption rate of 76% and its primary use case is Diagnostic Assistance.[1][2] |
| A | table reasoning | Which industry has the highest average ROI from AI? | True | 938 | 3.04 | Technology — 420% |
| B | table reasoning | Which industry has the highest average ROI from AI? | True | 987 | 2.68 | Technology has the highest average ROI from AI, at 420%. |
| C | table reasoning | Which industry has the highest average ROI from AI? | True | 506 | 5.47 | Technology has the highest average ROI from AI at 420%.[1][5] |
| A | cross-page table | What is Google Cloud's market share and YoY growth? | True | 940 | 3.62 | Google Cloud: 11% market share, +22% YoY growth. |
| B | cross-page table | What is Google Cloud's market share and YoY growth? | True | 989 | 2.83 | Google Cloud's market share is 11% and its YoY growth is +22%. |
| C | cross-page table | What is Google Cloud's market share and YoY growth? | True | 483 | 8.62 | Google Cloud’s market share is 11% and its YoY growth is +22%.[2] |
| A | cross-page table reasoning | Which cloud provider has the fastest year-over-year growth? | True | 939 | 2.99 | Google Cloud (+22%). |
| B | cross-page table reasoning | Which cloud provider has the fastest year-over-year growth? | True | 988 | 2.48 | Google Cloud has the fastest year-over-year growth at +22%. |
| C | cross-page table reasoning | Which cloud provider has the fastest year-over-year growth? | True | 568 | 4.95 | Google Cloud has the fastest year-over-year growth at 22% [1][2]. |
| A | table | What are Microsoft Azure's market share and key strength? | True | 939 | 2.52 | Microsoft Azure: 25% market share; Enterprise integration. |
| B | table | What are Microsoft Azure's market share and key strength? | True | 988 | 3.16 | Microsoft Azure has a market share of 25% and its key strength is enterprise integration. |
| C | table | What are Microsoft Azure's market share and key strength? | True | 547 | 4.68 | Microsoft Azure has a 25% market share and its key strength is enterprise integration. [1] |
| A | text | What does the report say about Kubernetes? | True | 936 | 2.5 | Kubernetes has solidified its position as the de facto standard for container orchestration. |
| B | text | What does the report say about Kubernetes? | True | 985 | 2.65 | Kubernetes has solidified its position as the de facto standard for container orchestration. |
| C | text | What does the report say about Kubernetes? | True | 660 | 4.68 | The report says Kubernetes has solidified its position as the de facto standard for container orchestration. [1] |
| A | text | Why has RAG become the preferred architecture for enterprise AI? | True | 940 | 2.81 | Because it combines the fluency of generative models with the accuracy of information retrieval. |
| B | text | Why has RAG become the preferred architecture for enterprise AI? | True | 989 | 2.84 | Because it combines the fluency of generative models with the accuracy of information retrieval. |
| C | text | Why has RAG become the preferred architecture for enterprise AI? | True | 729 | 5.07 | RAG has become preferred because it combines the fluency of generative models with the accuracy of information retrieval, making enterprise AI outputs more reliable. [1] |
| A | text | What does SBOM stand for and why is it mentioned? | True | 940 | 3.11 | SBOM stands for Software Bill of Materials. It is mentioned because supply chain security has emerged as a critical concern following high-profile incidents, and SBOM requirements are becoming standard. |
| B | text | What does SBOM stand for and why is it mentioned? | True | 989 | 3.19 | SBOM stands for Software Bill of Materials. It is mentioned in the cybersecurity section because SBOM requirements are becoming standard as part of supply chain security and stricter control over third-party dependencies. |
| C | text | What does SBOM stand for and why is it mentioned? | True | 712 | 4.73 | SBOM stands for Software Bill of Materials.[1] It is mentioned because SBOM requirements are becoming standard as part of growing supply chain security concerns, along with stricter controls over third-party dependencies.[1] |
| A | list | What is the first recommendation in the report? | True | 937 | 2.99 | Invest in RAG-based AI architectures for knowledge management applications. |
| B | list | What is the first recommendation in the report? | True | 986 | 2.97 | Invest in RAG-based AI architectures for knowledge management applications. |
| C | list | What is the first recommendation in the report? | True | 644 | 4.7 | Invest in RAG-based AI architectures for knowledge management applications. [2] |
| A | not in document | What is Oracle Cloud's market share? | True | 936 | 2.82 | The document does not say. |
| B | not in document | What is Oracle Cloud's market share? | True | 985 | 3.0 | The document does not say. |
| C | not in document | What is Oracle Cloud's market share? | True | 546 | 4.64 | The document does not say. |

### 9.4 One-time ingestion usage
What it cost to make the document searchable: CU pages, contextualization and LLM tokens (from `poller.usage`) plus index embeddings.

In [40]:
cu_usage = cu_meta["usage"]
cu_tokens = cu_usage.get("tokens") or {}
ingestion = {
    "pages_billed": sum(cu_usage.get(k, 0) or 0 for k in ("documentPagesMinimal", "documentPagesBasic", "documentPagesStandard")),
    "contextualization_tokens": cu_usage.get("contextualizationTokens", 0) or 0,
    "cu_llm_input_tokens": sum(v for k, v in cu_tokens.items() if "input" in k.lower()),
    "cu_llm_output_tokens": sum(v for k, v in cu_tokens.items() if "output" in k.lower()),
    "index_embedding_tokens": TOKEN_LOG["index_embedding_tokens"],
}
show_table([ingestion])
print("Raw CU token breakdown:", json.dumps(cu_tokens, indent=2))

| pages_billed | contextualization_tokens | cu_llm_input_tokens | cu_llm_output_tokens | index_embedding_tokens |
| --- | --- | --- | --- | --- |
| 3 | 3000 | 10020 | 2958 | 1831 |

Raw CU token breakdown: {
  "gpt-5.4-mini-input": 10020,
  "gpt-5.4-mini-output": 2958
}


### 9.5 Cost projection
Fill `PRICES` from the [Content Understanding pricing](https://azure.microsoft.com/pricing/details/content-understanding/) and [Azure OpenAI pricing](https://azure.microsoft.com/pricing/details/cognitive-services/openai-service/) pages for your region and deployment type. The zeros are placeholders, **not** prices.

Assumptions: A and B input tokens grow linearly with page count; C stays flat per query (top-k is fixed). B and C pay CU ingestion once (B: content extraction only). Query embeddings for C are ignored (tiny).

In [41]:
PRICES = {  # USD
    "cu_per_page": 0.0,
    "cu_contextualization_per_1m": 0.0,
    "mini_input_per_1m": 0.0,
    "mini_output_per_1m": 0.0,
    "premium_input_per_1m": 0.0,
    "premium_output_per_1m": 0.0,
    "embedding_per_1m": 0.0,
}
SCENARIO_PAGES = [doc.end_page_number - doc.start_page_number + 1, 300]
QUERY_COUNTS = [1, 100, 1_000, 10_000]


def per_1m(tokens: float, price: float) -> float:
    return tokens / 1_000_000 * price


pages_now = SCENARIO_PAGES[0]
ingest_per_page = (
    ingestion["pages_billed"] * PRICES["cu_per_page"]
    + per_1m(ingestion["contextualization_tokens"], PRICES["cu_contextualization_per_1m"])
    + per_1m(ingestion["cu_llm_input_tokens"], PRICES["mini_input_per_1m"])
    + per_1m(ingestion["cu_llm_output_tokens"], PRICES["mini_output_per_1m"])
    + per_1m(ingestion["index_embedding_tokens"], PRICES["embedding_per_1m"])
) / pages_now


def query_cost(key: str, pages: int) -> float:
    s = summary[key]
    tier = "premium" if key == "A" else "mini"
    input_tokens = s["avg_input_tokens"] * (1 if key == "C" else pages / pages_now)
    return per_1m(input_tokens, PRICES[f"{tier}_input_per_1m"]) + per_1m(s["avg_output_tokens"], PRICES[f"{tier}_output_per_1m"])


projection = []
for pages in SCENARIO_PAGES:
    for n in QUERY_COUNTS:
        projection.append(
            {
                "pages": pages,
                "queries": n,
                "A total $": round(n * query_cost("A", pages), 4),
                "B total $": round(pages * PRICES["cu_per_page"] + n * query_cost("B", pages), 4),
                "C total $": round(pages * ingest_per_page + n * query_cost("C", pages), 4),
                "A input tokens/query": round(summary["A"]["avg_input_tokens"] * pages / pages_now),
                "C input tokens/query": summary["C"]["avg_input_tokens"],
            }
        )
show_table(projection)

| pages | queries | A total $ | B total $ | C total $ | A input tokens/query | C input tokens/query |
| --- | --- | --- | --- | --- | --- | --- |
| 3 | 1 | 0.0 | 0.0 | 0.0 | 939 | 590 |
| 3 | 100 | 0.0 | 0.0 | 0.0 | 939 | 590 |
| 3 | 1000 | 0.0 | 0.0 | 0.0 | 939 | 590 |
| 3 | 10000 | 0.0 | 0.0 | 0.0 | 939 | 590 |
| 300 | 1 | 0.0 | 0.0 | 0.0 | 93900 | 590 |
| 300 | 100 | 0.0 | 0.0 | 0.0 | 93900 | 590 |
| 300 | 1000 | 0.0 | 0.0 | 0.0 | 93900 | 590 |
| 300 | 10000 | 0.0 | 0.0 | 0.0 | 93900 | 590 |

### 9.6 Takeaways (fill in after your run)
- **Accuracy:** did C match A, especially on the cross-page table questions?
- **Tokens per question:** A vs. C input tokens — and how the gap grows with page count.
- **Latency:** C adds a retrieval hop but uses a smaller model and context.
- **Where A is fine:** one short document and a handful of questions. CU + Search pays off with many documents, many questions, or documents larger than a context window.
- **Beyond Q&A:** the CU fields (`row_json`, confidence, grounding) feed workflows and analytics directly, with no LLM call at query time.

## 10. Clean up (optional) and next steps

In [ ]:
# index_client.delete_index(INDEX_NAME)
# cu_client.delete_analyzer(ANALYZER_ID)

**Ideas to explore next**
- Images, audio and video: `prebuilt-imageSearch`, `prebuilt-audioSearch`, `prebuilt-videoSearch`, or custom analyzers on `prebuilt-image` / `prebuilt-audio` / `prebuilt-video`.
- Multi-document files: `contentCategories` + `enableSegment` to split and route by document type.
- Let Azure AI Search ingest for you with the Content Understanding skill in an indexer (needs Basic tier or higher for managed identity).
- Agentic retrieval / Foundry IQ knowledge bases on top of this index.
- Improve field quality with labeled samples in Content Understanding Studio.